# Fatigue Prediction Model (RandomForest)

This notebook trains a Random Forest classifier to predict `fatigue_label` evaluating how close a user is to unsubscribing or tuning out.

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="darkgrid")

## 1. Load Data & Explore

In [ ]:
data_path = os.path.join("..", "processed", "fatigue_ready.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
display(df.head())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='fatigue_label', palette='magma')
plt.title('Target Variable Distribution (fatigue_label)')
plt.show()

## 2. Preprocessing & Encoding

In [ ]:
X = df[['previous_sends', 'ignored_count', 'unread_messages', 'rage_clicks']]
y = df['fatigue_label'].astype(str)

le_fatigue = LabelEncoder()
y_encoded = le_fatigue.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"Training set size: {len(X_train)} | Test set size: {len(X_test)}")

## 3. Train RandomForest Classifier

In [ ]:
model = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

## 4. Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le_fatigue.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd', xticklabels=le_fatigue.classes_, yticklabels=le_fatigue.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 5. Feature Importance

In [ ]:
feat_imps = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x='Importance', y='Feature', data=feat_imps, palette='viridis')
plt.title('RandomForest Feature Importance')
plt.show()

## 6. Export Model Bundle

In [ ]:
models_dir = os.path.join("..", "..", "backend", "models")
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, "fatigue.pkl")

bundle = {
    'model': model,
    'le_fatigue': le_fatigue,
    'features': list(X.columns)
}
joblib.dump(bundle, model_path)
print(f"Saved Model successfully to {model_path}")